# Day 23: Multiple Linear Regression

**Dataset:** `House_Prices_Dataset.csv`

This notebook builds a multiple linear regression model using several numerical and encoded
predictors to estimate a continuous target — a house's `price` — using real values pulled
directly from the House Prices dataset.

## 1. Import Libraries and Load the Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the dataset
df = pd.read_csv("../datasets/House_Prices_Dataset.csv")

# Preview the data
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [2]:
print(df.shape)
df.info()

(545, 13)
<class 'pandas.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   price             545 non-null    int64
 1   area              545 non-null    int64
 2   bedrooms          545 non-null    int64
 3   bathrooms         545 non-null    int64
 4   stories           545 non-null    int64
 5   mainroad          545 non-null    str  
 6   guestroom         545 non-null    str  
 7   basement          545 non-null    str  
 8   hotwaterheating   545 non-null    str  
 9   airconditioning   545 non-null    str  
 10  parking           545 non-null    int64
 11  prefarea          545 non-null    str  
 12  furnishingstatus  545 non-null    str  
dtypes: int64(6), str(7)
memory usage: 55.5 KB


## 2. Encoding the Categorical Predictors

`price`, `area`, `bedrooms`, `bathrooms`, `stories`, and `parking` are already numeric.
`mainroad`, `guestroom`, `basement`, `hotwaterheating`, `airconditioning`, and `prefarea` are
`yes`/`no` text columns, and `furnishingstatus` has three text categories — all of these need
to be encoded into numbers before a regression model can use them.

In [3]:
binary_columns = ["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea"]

for column in binary_columns:
    df[column] = df[column].map({"yes": 1, "no": 0})

df[binary_columns].head()

,mainroad,guestroom,basement,hotwaterheating,airconditioning,prefarea
0,1,0,0,0,1,1
1,1,0,0,0,1,0
2,1,0,1,0,0,1
3,1,0,1,0,1,1
4,1,1,1,0,1,0


**`furnishingstatus` has three categories, so it's one-hot encoded (with `drop_first=True` to avoid redundant columns)**

In [4]:
print(df["furnishingstatus"].unique())

df = pd.get_dummies(df, columns=["furnishingstatus"], drop_first=True)
df.head()

<StringArray>
['furnished', 'semi-furnished', 'unfurnished']
Length: 3, dtype: str


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,False,False
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,False,False
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,True,False
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,False,False
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,False,False


## 3. Separating Predictors and Target

In [5]:
X = df.drop(columns=["price"])
y = df["price"]

print("Predictor columns:", list(X.columns))
print("Target column: price")
print("X shape:", X.shape)
print("y shape:", y.shape)

Predictor columns: ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus_semi-furnished', 'furnishingstatus_unfurnished']
Target column: price
X shape: (545, 13)
y shape: (545,)


## 4. Checking Feature Correlations

In [6]:
df.corr()["price"].sort_values(ascending=False)

price                              1.000000
area                               0.535997
bathrooms                          0.517545
airconditioning                    0.452954
stories                            0.420712
parking                            0.384394
bedrooms                           0.366494
prefarea                           0.329777
mainroad                           0.296898
guestroom                          0.255517
basement                           0.187057
hotwaterheating                    0.093073
furnishingstatus_semi-furnished    0.063656
furnishingstatus_unfurnished      -0.280587
Name: price, dtype: float64

**Interpretation:** `area` has the strongest correlation with `price` (≈0.54), followed by
`bathrooms` (≈0.52) and `airconditioning` (≈0.45). `furnishingstatus_unfurnished` has a
meaningful negative correlation (≈-0.28) — unfurnished houses tend to sell for less, which
makes intuitive sense.

## 5. Train-Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

Training set size: 436
Test set size: 109


## 6. Building the Multiple Linear Regression Model

In [8]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained.")

Model trained.


## 7. Model Coefficients

In [9]:
coefficients = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_
}).sort_values("coefficient", key=abs, ascending=False)

print("Intercept:", model.intercept_)
coefficients

Intercept: 260032.3576074103


,feature,coefficient
2,bathrooms,1.094445e+06
8,airconditioning,7.914267e+05
7,hotwaterheating,6.846499e+05
10,prefarea,6.298906e+05
12,furnishingstatus_unfurnished,-4.136451e+05
3,stories,4.074766e+05
6,basement,3.902512e+05
4,mainroad,3.679199e+05
5,guestroom,2.316100e+05
9,parking,2.248419e+05


## 8. Prediction Results

In [10]:
y_pred = model.predict(X_test)

predictions_df = pd.DataFrame({
    "actual_price": y_test.values,
    "predicted_price": y_pred.round(0)
}).reset_index(drop=True)

predictions_df.head(10)

,actual_price,predicted_price
0,4060000,5164654.0
1,6650000,7224722.0
2,3710000,3109863.0
3,6440000,4612075.0
4,2800000,3294646.0
5,4900000,3532275.0
6,5250000,5611775.0
7,4543000,6368146.0
8,2450000,2722857.0
9,3353000,2629406.0


## 9. Train-Test Evaluation

Evaluating the model on the **test set** (data it never saw during training) is what
determines whether it actually generalizes.

In [11]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:,.0f}")
print(f"Root Mean Squared Error (RMSE): {rmse:,.0f}")
print(f"R-squared (R2): {r2:.3f}")

Mean Absolute Error (MAE): 970,043
Root Mean Squared Error (RMSE): 1,324,507
R-squared (R2): 0.653


**Comparing training-set performance to test-set performance, to check for overfitting**

In [12]:
y_train_pred = model.predict(X_train)
train_r2 = r2_score(y_train, y_train_pred)

print(f"Training R2: {train_r2:.3f}")
print(f"Test R2:     {r2:.3f}")
print("A similar R2 on both sets suggests the model generalizes reasonably well,")
print("rather than overfitting to the training data.")

Training R2: 0.686
Test R2:     0.653
A similar R2 on both sets suggests the model generalizes reasonably well,
rather than overfitting to the training data.


## 10. Interpretation of Important Features

Based on the coefficients above:

- **`bathrooms`** has the largest coefficient (≈+1,094,000) — each additional bathroom is
  associated with roughly a 1.09 million increase in predicted price, holding everything else
  constant. This is the single strongest driver of price in the model.
- **`airconditioning`** (≈+791,000) and **`hotwaterheating`** (≈+685,000) are the next
  strongest positive effects — homes with these amenities are predicted to be worth
  substantially more, even after accounting for size and room counts.
- **`area`** has a comparatively tiny coefficient (≈+236 per square foot) — but that's a scale
  effect, not a sign it doesn't matter: `area` ranges into the thousands, so its small
  per-unit coefficient still represents a large total contribution to price (adding 1,000
  square feet contributes roughly +236,000 to the predicted price). This is a good reminder
  that **coefficient size alone can be misleading when predictors are on very different
  scales** — `area`'s raw correlation with price (≈0.54) is actually the strongest of any
  feature, even though its coefficient looks small next to `bathrooms`.
- **`furnishingstatus_unfurnished`** has a negative coefficient (≈-414,000), consistent with
  its negative correlation with price — unfurnished homes are predicted to be worth less than
  furnished ones, all else being equal.

## Outcome

This notebook built a multiple linear regression model using `scikit-learn` to predict house
`price` from the `House_Prices_Dataset.csv` file, encoding six `yes`/`no` columns as 1/0 and
one-hot encoding the three-category `furnishingstatus` column before training. The model
reached a test R² of about **0.653** (versus 0.686 on the training set, suggesting reasonable
generalization rather than overfitting), an MAE of roughly 970,000, and an RMSE of roughly
1,325,000. Interpreting the coefficients showed `bathrooms`, `airconditioning`, and
`hotwaterheating` as the strongest positive drivers of price, while also revealing an
important nuance: `area` had the strongest raw correlation with price of any feature, but its
regression coefficient looked small purely because of its much larger numeric scale compared
to binary features like `bathrooms` — a useful reminder that a coefficient's size, on its own,
doesn't tell the whole story about how important a feature really is.